# EDA

## Setup

In [ ]:
import ast
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("../data") if Path("../data").exists() else Path("data")
pd.set_option("display.max_colwidth", 200)

## Load

In [ ]:
train = pd.read_csv(DATA / "uk_geo_dataset.csv")
processed = pd.read_parquet(DATA / "uk_geo_dataset_processed_v1.parquet")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "labeling_sample.csv")

frames = {"train": train, "processed": processed, "test": test, "sample": sample}
{name: df.shape for name, df in frames.items()}

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
sample.head()

## Overview

In [ ]:
pd.DataFrame({
    name: {
        "rows": len(df),
        "columns": list(df.columns),
        "nulls": int(df.isna().sum().sum()),
        "duplicate_texts": int(df["text"].duplicated().sum()) if "text" in df else None,
    }
    for name, df in frames.items()
}).T

In [ ]:
train["is_valid"].value_counts()

In [ ]:
test["locations"].value_counts()

## Entity markers

In [ ]:
SPAN = re.compile(r"\((\d+),\s*(\d+)\)")


def parse_spans(value):
    return [(int(a), int(b)) for a, b in SPAN.findall(str(value))]


for col in ["loc_markers", "org_markers", "per_markers"]:
    train[col] = train[col].map(parse_spans)
sample["loc_markers"] = sample["loc_markers"].map(parse_spans)
sample["locations"] = sample["locations"].map(ast.literal_eval)

counts = pd.DataFrame({e: train[f"{e}_markers"].map(len) for e in ["loc", "org", "per"]})
pd.DataFrame({
    "texts_with_entity": (counts > 0).mean(),
    "mean_per_text": counts.mean(),
    "total": counts.sum(),
})

In [ ]:
(counts["loc"] > 0).groupby(train["is_valid"]).mean()

In [ ]:
counts["loc"].value_counts().sort_index().head(10)

## Location spans

In [ ]:
loc_strings = pd.Series(
    [text[a:b] for text, spans in zip(train["text"], train["loc_markers"]) for a, b in spans]
)
pd.DataFrame({
    "chars": loc_strings.str.len().describe(),
    "words": loc_strings.str.split().str.len().describe(),
})

In [ ]:
loc_counts = loc_strings.value_counts()
print("unique locations:", len(loc_counts))
print("mentions covered by top 100:", round(loc_counts.head(100).sum() / loc_counts.sum(), 3))
print("locations seen once:", int((loc_counts == 1).sum()))
loc_counts.head(30)

In [ ]:
def span_issues(text, spans):
    issues = []
    for a, b in spans:
        if not 0 <= a < b <= len(text):
            issues.append("out_of_bounds")
        elif text[a:b] != text[a:b].strip():
            issues.append("edge_whitespace")
    ordered = sorted(spans)
    issues += ["overlap" for (a1, b1), (a2, b2) in zip(ordered, ordered[1:]) if a2 < b1]
    return issues


Counter(i for text, spans in zip(train["text"], train["loc_markers"]) for i in span_issues(text, spans))

In [ ]:
def overlaps(spans_a, spans_b):
    return any(a1 < b2 and a2 < b1 for a1, b1 in spans_a for a2, b2 in spans_b)


pd.Series({
    "loc_overlaps_org": sum(overlaps(l, o) for l, o in zip(train["loc_markers"], train["org_markers"])),
    "loc_overlaps_per": sum(overlaps(l, p) for l, p in zip(train["loc_markers"], train["per_markers"])),
})

In [ ]:
def adjacent_pairs(text, spans):
    ordered = sorted(spans)
    return [(text[a1:b1], text[b1:a2], text[a2:b2]) for (a1, b1), (a2, b2) in zip(ordered, ordered[1:]) if text[b1:a2].strip() == ""]


adjacent = [pair for text, spans in zip(train["text"], train["loc_markers"]) for pair in adjacent_pairs(text, spans)]
print("adjacent location pairs separated only by whitespace:", len(adjacent))
adjacent[:10]

## Processed tokens and labels

In [ ]:
processed["tokens"] = processed["tokens"].map(json.loads)
processed["labels"] = processed["labels"].map(json.loads)

tag_counts = Counter(tag for labels in processed["labels"] for tag in labels)
pd.Series(tag_counts).to_frame("count").assign(share=lambda d: d["count"] / d["count"].sum())

In [ ]:
def count_runs(labels, tag="LOC"):
    return sum(1 for i, label in enumerate(labels) if label == tag and (i == 0 or labels[i - 1] != tag))


processed_loc = processed["labels"].map(count_runs)
print("same is_valid order:", bool((processed["is_valid"].values == train["is_valid"].values).all()))
print("rows where LOC run count matches loc_markers:", round(float((processed_loc.values == counts["loc"].values).mean()), 4))
mismatch = np.flatnonzero(processed_loc.values != counts["loc"].values)
train.iloc[mismatch[:5]][["text", "loc_markers"]].assign(processed_runs=processed_loc.values[mismatch[:5]])

In [ ]:
token_lengths = processed["tokens"].map(len)
token_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

## Train vs test domain

In [ ]:
EMOJI = "[\U0001F000-\U0001FAFF☀-➿]"


def text_profile(texts):
    length = texts.str.len().clip(lower=1)
    return pd.DataFrame({
        "chars": texts.str.len(),
        "lines": texts.str.count("\n") + 1,
        "has_emoji": texts.str.contains(EMOJI),
        "has_url": texts.str.contains(r"https?://|www\.|t\.me/"),
        "has_hashtag_or_mention": texts.str.contains(r"[#@]\w"),
        "has_russian_letters": texts.str.contains("[ыэъёЫЭЪЁ]"),
        "latin_share": texts.str.count("[A-Za-z]") / length,
        "upper_share": texts.str.count("[A-ZА-ЯІЇЄҐ]") / length,
    })


train_texts = train["text"].sample(100_000, random_state=0)
pd.DataFrame({
    "train": text_profile(train_texts).mean(),
    "test": text_profile(test["text"]).mean(),
    "sample": text_profile(sample["text"]).mean(),
}).round(3)

In [ ]:
TOKEN = re.compile(r"\w+(?:['’ʼ-]\w+)*|[^\w\s]")


def tokenize(text):
    return TOKEN.findall(text)


lengths = {
    "train": token_lengths.sample(100_000, random_state=0),
    "test": test["text"].map(lambda t: len(tokenize(t))),
    "sample": sample["text"].map(lambda t: len(tokenize(t))),
}
pd.DataFrame({name: values.describe(percentiles=[0.5, 0.9, 0.95, 0.99]) for name, values in lengths.items()}).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 400, 81)
for name, values in lengths.items():
    ax.hist(values.clip(upper=400), bins=bins, density=True, histtype="step", label=name)
ax.set_xlabel("tokens per text")
ax.set_ylabel("density")
ax.legend()
plt.show()

In [ ]:
train_vocab = Counter(token.lower() for tokens in processed["tokens"] for token in tokens)
frequent_vocab = {token for token, n in train_vocab.items() if n >= 5}


def oov_report(texts):
    words = [token.lower() for text in texts for token in tokenize(text) if token[0].isalpha()]
    return {
        "words": len(words),
        "oov_full_vocab": round(np.mean([w not in train_vocab for w in words]), 4),
        "oov_freq_ge_5": round(np.mean([w not in frequent_vocab for w in words]), 4),
    }


print("train vocabulary:", len(train_vocab), "| with frequency >= 5:", len(frequent_vocab))
pd.DataFrame({"test": oov_report(test["text"]), "sample": oov_report(sample["text"])})

In [ ]:
Counter(
    token.lower() for text in test["text"] for token in tokenize(text)
    if token[0].isalpha() and token.lower() not in train_vocab
).most_common(30)

## Labeling sample

In [ ]:
print("locations match markers:", all(
    [text[a:b] for a, b in spans] == locations
    for text, spans, locations in zip(sample["text"], sample["loc_markers"], sample["locations"])
))
sample_counts = sample["locations"].map(len)
print("texts with a location:", round(float((sample_counts > 0).mean()), 3))
sample_counts.describe()

In [ ]:
sample_locations = pd.Series([loc for locs in sample["locations"] for loc in locs])
train_locations = set(loc_counts.index.str.lower())
print("sample locations seen in train:", round(float(sample_locations.str.lower().isin(train_locations).mean()), 3))
sample_locations.to_frame("location").assign(in_train=sample_locations.str.lower().isin(train_locations))

## Metric

In [ ]:
import string
from typing import List


def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p,  = 0.0, 0.0, 0.0, 0.0

    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])

        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)

    if tp + fp == 0:
        if p == 0:
            precision = 1.0
        else:
            precision = 0.0
    else:
        precision = tp/(tp + fp)

    if tp + fn == 0:
        if p == 0:
            recall = 1.0
        else:
            recall = 0.0
    else:
        recall = tp/(tp + fn)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

In [ ]:
pd.DataFrame(
    [(s, hard_processing(s)) for s in [
        "Київ", "м. Київ", "(Київ)", "Київ!", "Київ, Україна", "Івано-Франківськ",
        "вул. Солом'янська, 28", "Kyiv", "KYIV", "Nature reserve", "В", "1",
    ]],
    columns=["raw", "processed"],
)

In [ ]:
REMOVED = set(r"{re.escape(string.punctuation)}")
print("characters removed by the punctuation step:", "".join(sorted(REMOVED)))
print("punctuation kept:", "".join(c for c in string.punctuation if c not in REMOVED))

In [ ]:
processed_locs = pd.Series(loc_counts.index.map(hard_processing), index=loc_counts.index)
pd.Series({
    "unique_train_locations": len(loc_counts),
    "unique_after_processing": processed_locs.nunique(),
    "mentions_processed_to_empty": int(loc_counts[processed_locs == ""].sum()),
    "mentions_with_latin_damage": int(loc_counts[loc_counts.index.str.contains("[aceginoprstu]")].sum()),
})

In [ ]:
y_true = sample["locations"].tolist()


def first_word(locations):
    return [loc.split()[0] for loc in locations]


pd.Series({
    "perfect": comp_metric(y_true, y_true),
    "all_empty": comp_metric(y_true, [[] for _ in y_true]),
    "lowercased": comp_metric(y_true, [[loc.lower() for loc in locs] for locs in y_true]),
    "with_prefix_m": comp_metric(y_true, [["м. " + loc for loc in locs] for locs in y_true]),
    "first_word_only": comp_metric(y_true, [first_word(locs) for locs in y_true]),
    "duplicated": comp_metric(y_true, [locs + locs for locs in y_true]),
    "plus_one_wrong": comp_metric(y_true, [locs + ["Марс"] for locs in y_true]),
})

## Train/test overlap

In [ ]:
train_set = set(train["text"])
pd.Series({
    "test_texts_in_train": int(test["text"].isin(train_set).sum()),
    "sample_texts_in_train": int(sample["text"].isin(train_set).sum()),
    "sample_texts_in_test": int(sample["text"].isin(set(test["text"])).sum()),
})